# Solution 05 — Ambiguous IT Ticket Resolver

Reference implementation for Assessment 05.


## Architectural Decisions

Pipeline: **intent decompose → prioritize/route → slot-fill with directory → plan tools → draft reply**

1. Extract distinct intents from the noisy message.
2. Prioritize / mark deferred given user language.
3. Bind entities (asset_id, manager) from directory; list blocking clarifications.
4. Propose tool calls + user reply in one structured planning call (or split plan/reply).

This scenario stresses **routing + clarification policy**, unlike evidence auditing (04) or arithmetic validation (03).


In [ ]:
import json
import os
import re
from typing import Any, Optional

# ---------------------------------------------------------------------------
# LLM configuration
# ---------------------------------------------------------------------------
# Set one of the following before running (or export in your shell):
#
#   export OPENAI_API_KEY="sk-..."
#   # optional: export OPENAI_MODEL="gpt-4o-mini"
#
#   export ANTHROPIC_API_KEY="sk-ant-..."
#   # optional: export ANTHROPIC_MODEL="claude-3-5-haiku-latest"
#
# If neither key is set, call_llm runs in MOCK mode so the notebook still
# executes for smoke-testing. Mock responses are placeholders — they are not
# a substitute for a real model during practice.
# ---------------------------------------------------------------------------

OPENAI_API_KEY = os.environ.get("OPENAI_API_KEY", "").strip()
ANTHROPIC_API_KEY = os.environ.get("ANTHROPIC_API_KEY", "").strip()
OPENAI_MODEL = os.environ.get("OPENAI_MODEL", "gpt-4o-mini")
ANTHROPIC_MODEL = os.environ.get("ANTHROPIC_MODEL", "claude-3-5-haiku-latest")

USE_MOCK = not (OPENAI_API_KEY or ANTHROPIC_API_KEY)
if USE_MOCK:
    print("⚠️  No API key found. call_llm is in MOCK mode.")
else:
    provider = "OpenAI" if OPENAI_API_KEY else "Anthropic"
    print(f"✓ Using {provider} API")


def _mock_response(prompt: str, system_prompt: str = "") -> str:
    """Lightweight deterministic mock for offline smoke tests."""
    combined = f"{system_prompt}\n{prompt}".lower()
    # Return minimal JSON-ish or text so downstream parsing can be exercised.
    if "json" in combined:
        return json.dumps({"mock": True, "note": "Replace with real API key for useful output", "items": []})
    return "[MOCK RESPONSE] Configure OPENAI_API_KEY or ANTHROPIC_API_KEY for real model output."


def call_llm(
    prompt: str,
    system_prompt: str = "You are a helpful assistant.",
    temperature: float = 0.2,
    max_tokens: int = 2000,
) -> str:
    """Call a chat LLM. Credentials come from environment variables only."""
    if USE_MOCK:
        return _mock_response(prompt, system_prompt)

    try:
        if OPENAI_API_KEY:
            from openai import OpenAI

            client = OpenAI(api_key=OPENAI_API_KEY)
            resp = client.chat.completions.create(
                model=OPENAI_MODEL,
                temperature=temperature,
                max_tokens=max_tokens,
                messages=[
                    {"role": "system", "content": system_prompt},
                    {"role": "user", "content": prompt},
                ],
            )
            return (resp.choices[0].message.content or "").strip()

        from anthropic import Anthropic

        client = Anthropic(api_key=ANTHROPIC_API_KEY)
        resp = client.messages.create(
            model=ANTHROPIC_MODEL,
            max_tokens=max_tokens,
            temperature=temperature,
            system=system_prompt,
            messages=[{"role": "user", "content": prompt}],
        )
        return "".join(
            block.text for block in resp.content if getattr(block, "type", None) == "text"
        ).strip()
    except Exception as e:
        return f"[API ERROR] {type(e).__name__}: {e}"


def extract_json(text: str) -> Any:
    """Best-effort parse of a JSON object/array from model output."""
    text = text.strip()
    try:
        return json.loads(text)
    except json.JSONDecodeError:
        pass
    fence = re.search(r"```(?:json)?\s*([\s\S]*?)```", text)
    if fence:
        try:
            return json.loads(fence.group(1).strip())
        except json.JSONDecodeError:
            pass
    for opener, closer in (("{", "}"), ("[", "]")):
        start, end = text.find(opener), text.rfind(closer)
        if start != -1 and end > start:
            try:
                return json.loads(text[start : end + 1])
            except json.JSONDecodeError:
                continue
    raise ValueError(f"Could not parse JSON from model output:\n{text[:500]}")


# Smoke test
print("call_llm ready.")
print(call_llm("Reply with exactly: OK", system_prompt="Be terse.")[:80])


In [ ]:
EMPLOYEE_DIRECTORY = [
  {
    "user_id": "u_88",
    "name": "Sam Ortiz",
    "dept": "Finance",
    "manager": "u_12",
    "laptop_asset": "L-2031"
  },
  {
    "user_id": "u_19",
    "name": "Alex Kim",
    "dept": "Engineering",
    "manager": "u_02",
    "laptop_asset": "L-7781"
  }
]

TICKET = {
  "ticket_id": "IT-55401",
  "requester_id": "u_19",
  "subject": "urgent cant access + need software",
  "body": "hey my vpn keeps failing after the password reset yesterday and also can someone install the design plugin on my machine before the client demo tomorrow? or wait maybe the demo is Thursday. also I think my MFA soft token is weird. If vpn is a bigger fire ignore the plugin. thanks",
  "channel": "slack_dm",
  "created_at": "2026-10-13T16:05:00Z"
}

TOOL_CATALOG = """Available resolution tools (you cannot execute them; propose calls only):
1. reset_vpn_profile(user_id)
2. push_mfa_reenrollment(user_id)
3. install_software(asset_id, package_name)
4. schedule_technician(asset_id, window)
5. request_manager_approval(manager_id, reason)
6. reply_to_user(ticket_id, message)"""


## Why Each Call

| Call | Role |
|------|------|
| Decompose intents | Multi-intent unpacking |
| Prioritize | Applies user's "if vpn is a bigger fire…" rule |
| Entity bind + clarifications | Directory grounding; minimize questions |
| Plan + reply | Tool policy compliance and UX |


In [ ]:
def resolve_ambiguous_ticket(ticket: dict, employee_directory: list, tool_catalog: str) -> dict:
    if not ticket:
        return {
            "intents": [],
            "clarifying_questions": ["Empty ticket"],
            "proposed_tool_calls": [],
            "execution_order": [],
            "user_reply": "We received an empty request — please resend.",
            "internal_notes": "empty input",
            "assumptions": [],
        }

    requester = next(
        (e for e in employee_directory if e.get("user_id") == ticket.get("requester_id")),
        None,
    )

    # Call 1: intent decomposition
    intents_raw = call_llm(
        prompt=(
            "Extract distinct IT intents from the ticket. Return JSON "
            "{\"intents\":[{name, raw_span, notes}]} where name is one of "
            "vpn_access|mfa_issue|software_install|other.\n\n"
            f"TICKET:\n{json.dumps(ticket, indent=2)}"
        ),
        system_prompt="You unpack multi-intent IT requests.",
        temperature=0.1,
    )
    try:
        intents = extract_json(intents_raw).get("intents", [])
    except ValueError:
        intents = []

    # Call 2: prioritize / defer
    pri_raw = call_llm(
        prompt=(
            "Assign priority p1|p2|p3 and status actionable|needs_clarification|deferred "
            "to each intent using the user's own priority language. Return JSON "
            "{\"intents\":[{name, priority, urgency_reason, status}]}\n\n"
            f"INTENTS:\n{json.dumps(intents)}\n\nTICKET_BODY:\n{ticket.get('body','')}"
        ),
        system_prompt="You prioritize IT work for automation safely.",
        temperature=0.0,
    )
    try:
        prioritized = extract_json(pri_raw).get("intents", [])
    except ValueError:
        prioritized = intents

    # Call 3: clarifications + entity binding needs
    bind_raw = call_llm(
        prompt=(
            "Using the employee record and prioritized intents, list blocking clarifying "
            "questions ONLY if required to call a tool (e.g., missing package name). "
            "Return JSON {\"clarifying_questions\":[...], \"assumptions\":[...], "
            "\"entity_context\":{...}}\n\n"
            f"EMPLOYEE:\n{json.dumps(requester)}\n\nINTENTS:\n{json.dumps(prioritized)}\n\n"
            f"TOOLS:\n{tool_catalog}"
        ),
        system_prompt="Minimize clarifying questions; prefer safe assumptions when stated.",
        temperature=0.1,
    )
    try:
        bind = extract_json(bind_raw)
    except ValueError:
        bind = {"clarifying_questions": [], "assumptions": [], "entity_context": {}}

    # Call 4: tool plan + user reply
    plan_raw = call_llm(
        prompt=(
            "Propose tool calls ONLY from the catalog, with args and depends_on. "
            "Include execution_order, user_reply, internal_notes. "
            "If software install is deferred, do not install yet. "
            "Return JSON with keys: proposed_tool_calls, execution_order, user_reply, "
            "internal_notes.\n\n"
            f"TICKET:\n{json.dumps(ticket)}\n"
            f"EMPLOYEE:\n{json.dumps(requester)}\n"
            f"INTENTS:\n{json.dumps(prioritized)}\n"
            f"CLARIFY:\n{json.dumps(bind)}\n"
            f"TOOLS:\n{tool_catalog}"
        ),
        system_prompt="You are an IT automation planner. Never invent tools.",
        temperature=0.1,
    )
    try:
        plan = extract_json(plan_raw)
    except ValueError:
        plan = {
            "proposed_tool_calls": [],
            "execution_order": [],
            "user_reply": "We're looking into your VPN issue first and will follow up on software.",
            "internal_notes": "plan parse failure",
        }

    # Normalize intent schema
    final_intents = []
    for row in prioritized:
        final_intents.append(
            {
                "name": row.get("name", "other"),
                "priority": row.get("priority", "p2"),
                "urgency_reason": row.get("urgency_reason", ""),
                "status": row.get("status", "actionable"),
            }
        )

    return {
        "intents": final_intents,
        "clarifying_questions": bind.get("clarifying_questions", []),
        "proposed_tool_calls": plan.get("proposed_tool_calls", []),
        "execution_order": plan.get("execution_order", []),
        "user_reply": plan.get("user_reply", ""),
        "internal_notes": plan.get("internal_notes", ""),
        "assumptions": bind.get("assumptions", []),
    }


## Example Execution


In [ ]:
result = resolve_ambiguous_ticket(TICKET, EMPLOYEE_DIRECTORY, TOOL_CATALOG)
print(json.dumps(result, indent=2)[:4000])


## Alternatives

- Single classify+reply call: under-handles multi-intent deferral.
- Always ask 5 clarifying questions: poor UX.
- Fully autonomous execution without approval notes: risky for interview scope.

## Tradeoffs

Extra priority call encodes user-stated preference explicitly. Clarification call reduces over-asking.

## Common Mistakes

- Installing software before VPN.
- Inventing tools.
- Ignoring directory asset id.
- Asking questions already answered.

## Beyond 45 Minutes

- Real tool execution sandbox; confidence thresholds; memory of prior tickets; eval on multi-intent corpus.
